# 🔧 05 — Optimisation des Hyperparamètres
RandomizedSearchCV → GridSearchCV → meilleur modèle XGBoost.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd, numpy as np, warnings; warnings.filterwarnings('ignore')
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

session = get_active_session()
session.sql("USE DATABASE HOUSE_PRICE_DB").collect()

FEATURE_COLS = [
    'AREA','BEDROOMS','BATHROOMS','STORIES','MAINROAD','GUESTROOM',
    'BASEMENT','HOTWATERHEATING','AIRCONDITIONING','PARKING','PREFAREA',
    'FURNISHING_ENCODED','AREA_BEDS','TOTAL_ROOMS','PREMIUM_FLAG'
]
train = session.table("ML.TRAIN_SET").to_pandas()
test = session.table("ML.TEST_SET").to_pandas()
X_train, y_train = train[FEATURE_COLS], train['PRICE']
X_test, y_test = test[FEATURE_COLS], test['PRICE']

In [ ]:
# Phase 1 : RandomizedSearchCV (exploration large)
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2],
}

rscv = RandomizedSearchCV(
    xgb.XGBRegressor(random_state=42, verbosity=0, n_jobs=-1),
    param_dist, n_iter=50, cv=5, scoring='r2', random_state=42, n_jobs=-1
)
rscv.fit(X_train, y_train)
print("RandomizedSearch meilleurs params:", rscv.best_params_)
print("RandomizedSearch meilleur CV-R²:", round(rscv.best_score_, 4))

In [ ]:
# Phase 2 : GridSearchCV (affinage fin autour des meilleurs params)
bp = rscv.best_params_
param_grid = {
    'n_estimators': [bp['n_estimators'] - 50, bp['n_estimators'], bp['n_estimators'] + 50],
    'max_depth': [max(2, bp['max_depth']-1), bp['max_depth'], bp['max_depth']+1],
    'learning_rate': [bp['learning_rate']],
    'subsample': [bp['subsample']],
    'colsample_bytree': [bp['colsample_bytree']],
    'min_child_weight': [bp['min_child_weight']],
}
gscv = GridSearchCV(
    xgb.XGBRegressor(random_state=42, verbosity=0, n_jobs=-1),
    param_grid, cv=5, scoring='r2', n_jobs=-1
)
gscv.fit(X_train, y_train)
best_model = gscv.best_estimator_
print("GridSearch meilleurs params:", gscv.best_params_)

pred = best_model.predict(X_test)
metrics = {
    "R²": round(r2_score(y_test, pred), 4),
    "RMSE": round(np.sqrt(mean_squared_error(y_test, pred)), 0),
    "MAE": round(mean_absolute_error(y_test, pred), 0),
}
print("Métriques finales XGBoost optimisé:", metrics)

In [ ]:
# Feature importance
import matplotlib.pyplot as plt

feat_imp = pd.Series(best_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
feat_imp.plot(kind='barh', figsize=(8,6), color='#01696f')
plt.title("Feature Importance — XGBoost Optimisé")
plt.tight_layout(); plt.show()